---
### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 3.1 — Adding Retrieval to the Agent



**Where we are:** Our MLflow assistant answers questions, but it struggles to gather context about MLflow 3.0.

In this notebook we'll:
1. Build a simple document store with MLflow 3 knowledge
2. Create a RAG pipeline from scratch — embedding, search, prompt construction
3. **Then** add MLflow Tracing to see inside the pipeline

We intentionally build the pipeline *without* tracing first, then layer it on — so you can see exactly what tracing adds.

---
## 0 — Environment Setup

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3-flash-preview"
EMBEDDING_MODEL = "text-embedding-004"

print(f"LLM: {MODEL}")
print(f"Embeddings: {EMBEDDING_MODEL}")

---
## 1 — The Document Store

In production this would be a vector database. For this workshop we use a plain dictionary — 10 documents covering core MLflow 3 capabilities that our assistant should be able to reference.

In [ ]:
DOCUMENT_STORE = {
    "doc1": (
        "MLflow Tracing provides end-to-end observability for GenAI applications. "
        "It captures every LLM call, retrieval step, tool invocation, and agent reasoning "
        "as structured spans with full input/output visibility. Traces are logged automatically "
        "via autologging or manually with the @mlflow.trace decorator and mlflow.start_span() context manager."
    ),
    "doc2": (
        "MLflow AI Gateway lets teams manage multiple LLM providers through a single, secure endpoint. "
        "It centralizes access control, cost tracking, and rate limiting across providers like OpenAI, "
        "Anthropic, and Google. Features include traffic routing, automatic fallbacks between providers, "
        "budget alerts, and comprehensive usage analytics."
    ),
    "doc3": (
        "MLflow Evaluation enables systematic testing of GenAI applications using scorers. "
        "Built-in scorers include Correctness, Safety, RelevanceToQuery, and RetrievalGroundedness. "
        "Teams can also create custom scorers with Guidelines() for natural-language criteria "
        "or make_judge() for LLM-as-judge evaluators. Run evaluations with mlflow.genai.evaluate()."
    ),
    "doc4": (
        "MLflow Prompt Registry allows teams to version, share, and manage prompts centrally. "
        "Prompts support template variables, lifecycle aliases (development, staging, production), "
        "and integration with experiments. Use mlflow.genai.register_prompt() to store prompts "
        "and mlflow.genai.load_prompt() to retrieve them by name and version."
    ),
    "doc5": (
        "Judge alignment in MLflow teaches LLM judges to match human evaluation standards. "
        "The workflow is: create a judge with make_judge(), collect human feedback on traces "
        "via mlflow.log_feedback(), then call judge.align() with the SIMBA or GEPA optimizer. "
        "Aligned judges typically show 30-50% reduction in false positives compared to generic prompts."
    ),
    "doc6": (
        "MLflow autologging automatically captures metrics, parameters, and traces for 40+ frameworks "
        "including OpenAI, LangChain, LlamaIndex, DSPy, and AutoGen. Enable it with one line — "
        "e.g., mlflow.openai.autolog() or mlflow.langchain.autolog(). Each call is traced with "
        "token counts, latency, and the full request/response payload."
    ),
    "doc7": (
        "MLflow Deployments serves models as production-ready REST API endpoints. "
        "It supports local inference servers, Kubernetes, AWS SageMaker, and managed platforms. "
        "Models are deployed directly from the Model Registry with version pinning, "
        "and each serving endpoint gets automatic request/response logging."
    ),
    "doc8": (
        "MLflow tracks token usage and cost across LLM applications. Every traced call records "
        "input tokens, output tokens, and total tokens. When provider pricing is configured, "
        "MLflow calculates per-call and cumulative costs. Use mlflow.search_traces() to query "
        "usage patterns and identify expensive operations."
    ),
    "doc9": (
        "MLflow Experiment Tracking organizes ML and GenAI work into experiments and runs. "
        "Each run logs parameters, metrics, artifacts, and traces. Teams use mlflow.set_experiment() "
        "to group related work, mlflow.log_param() and mlflow.log_metric() for tracking, "
        "and the MLflow UI to compare results across runs visually."
    ),
    "doc10": (
        "MLflow is fully open source and vendor-neutral. It works with any cloud provider, "
        "ML framework, or LLM provider without lock-in. The unified API covers the full lifecycle "
        "from experimentation through evaluation to production deployment, with a single tracking "
        "server that stores all artifacts, metrics, and traces."
    ),
}

print(f"Document store initialized with {len(DOCUMENT_STORE)} documents")
for doc_id, text in DOCUMENT_STORE.items():
    print(f"  {doc_id}: {text[:80]}...")

---
## 2 — Building the RAG Pipeline (without tracing)

A RAG pipeline has four stages:

| Stage | Function | What it does |
|---|---|---|
| **Embed** | `get_embeddings()` | Turn text into vectors |
| **Index** | `build_index()` | Embed all documents up front |
| **Search** | `semantic_search()` | Find the most relevant docs for a query |
| **Generate** | `rag_query()` | Build a grounded prompt and call the LLM |

We'll build each one as a plain Python function first.

### 2a — Embedding function

We use the Gemini embeddings API through the OpenAI-compatible endpoint. This returns a dense vector for each input text.

In [ ]:
def get_embeddings(texts: list[str]) -> list[list[float]]:
    """Get embeddings for a list of texts using the Gemini embedding model."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    return [item.embedding for item in response.data]

# Quick test
test_emb = get_embeddings(["hello world"])
print(f"Embedding dim: {len(test_emb[0])}")

### 2b — Build the document index

We embed every document once and store the vectors alongside the text. In production you'd use a vector DB — here a list of dicts is fine.

In [ ]:
def build_index(document_store: dict[str, str]) -> list[dict]:
    """Embed all documents and return an index of {id, text, embedding}."""
    doc_ids = list(document_store.keys())
    doc_texts = list(document_store.values())
    embeddings = get_embeddings(doc_texts)

    index = []
    for doc_id, text, emb in zip(doc_ids, doc_texts, embeddings):
        index.append({"id": doc_id, "text": text, "embedding": emb})

    return index

doc_index = build_index(DOCUMENT_STORE)
print(f"Indexed {len(doc_index)} documents")

### 2c — Semantic search

Given a query, embed it and find the `top_k` most similar documents using cosine similarity.

In [ ]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Compute cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def semantic_search(query: str, index: list[dict], top_k: int = 3) -> list[dict]:
    """Find the top_k most relevant documents for a query."""
    query_embedding = get_embeddings([query])[0]

    scored = []
    for doc in index:
        score = cosine_similarity(query_embedding, doc["embedding"])
        scored.append({"id": doc["id"], "text": doc["text"], "score": score})

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]

# Test it
results = semantic_search("How do I trace my LLM calls?", doc_index)
for r in results:
    print(f"  [{r['score']:.3f}] {r['id']}: {r['text'][:80]}...")

### 2d — Prompt construction

This is the "augmented" part of RAG — we inject the retrieved documents into the prompt so the LLM can ground its answer in real content.

In [ ]:
SYSTEM_PROMPT = """You are a helpful MLflow assistant. Answer the user's question using ONLY
the provided context documents. If the context doesn't contain enough information
to answer fully, say so honestly.

Context:
{context}"""


def build_prompt(question: str, retrieved_docs: list[dict]) -> list[dict]:
    """Construct the chat messages with retrieved context injected."""
    context = "\n\n".join(
        f"[{doc['id']}] {doc['text']}" for doc in retrieved_docs
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT.format(context=context)},
        {"role": "user", "content": question},
    ]

# Preview what the prompt looks like
sample_messages = build_prompt("How do I trace my LLM calls?", results)
print("System prompt (first 300 chars):")
print(sample_messages[0]["content"][:300])
print(f"\nUser: {sample_messages[1]['content']}")

### 2e — The complete RAG query function

Ties it all together: search -> build prompt -> call LLM.

In [ ]:
def rag_query(question: str, index: list[dict], top_k: int = 3) -> str:
    """End-to-end RAG: retrieve relevant docs, build prompt, generate answer."""
    # Retrieve
    retrieved_docs = semantic_search(question, index, top_k=top_k)

    # Augment
    messages = build_prompt(question, retrieved_docs)

    # Generate
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2,
        max_completion_tokens=400,
    )

    return response.choices[0].message.content

In [ ]:
# Try it out
answer = rag_query("How do I trace my LLM calls in MLflow?", doc_index)
print(answer)

In [ ]:
answer = rag_query("What is the AI Gateway and what does it do?", doc_index)
print(answer)

---
## 3 — The problem: it works, but it's a black box

The pipeline returns answers, but we can't see *inside* it:

- Which documents were retrieved? Were they relevant?
- How long did embedding take vs. generation?
- What exact prompt was sent to the LLM?
- How many tokens did it use?

Without observability, debugging is guesswork. Let's fix that with **MLflow Tracing**.

---
## 4 — Adding MLflow Tracing

We'll re-define the same functions, but now with tracing decorators. MLflow gives us two tools:

| Tool | Use when |
|---|---|
| `@mlflow.trace` | Decorating a function — captures inputs, outputs, and timing automatically |
| `mlflow.start_span()` | You need a span *inside* a function (e.g., around a loop or API call) |

Each span gets a `span_type` that tells the MLflow UI how to render it:

| Span type | Meaning |
|---|---|
| `EMBEDDING` | A vectorization call |
| `RETRIEVER` | A document lookup/search |
| `LLM` | A language model call |
| `CHAIN` | A multi-step orchestration |

In [ ]:
import mlflow
from mlflow.entities import SpanType

tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("mlflow-agent-3-retrieval")

print("MLflow tracing enabled")

### 4a — Traced embedding function

The `@mlflow.trace` decorator wraps the function in a span. We set `span_type=SpanType.EMBEDDING` so the UI knows this is a vectorization step.

In [ ]:
@mlflow.trace(span_type=SpanType.EMBEDDING)
def get_embeddings_traced(texts: list[str]) -> list[list[float]]:
    """Get embeddings for a list of texts (traced)."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    return [item.embedding for item in response.data]

### 4b — Traced semantic search

The retriever span captures the query and the documents it found — this is critical for debugging relevance issues.

In [ ]:
@mlflow.trace(span_type=SpanType.RETRIEVER)
def semantic_search_traced(query: str, index: list[dict], top_k: int = 3) -> list[dict]:
    """Find the top_k most relevant documents for a query (traced)."""
    query_embedding = get_embeddings_traced([query])[0]

    scored = []
    for doc in index:
        score = cosine_similarity(query_embedding, doc["embedding"])
        scored.append({"id": doc["id"], "text": doc["text"], "score": score})

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]

### 4c — Traced prompt construction

In [ ]:
@mlflow.trace(span_type=SpanType.CHAIN)
def build_prompt_traced(question: str, retrieved_docs: list[dict]) -> list[dict]:
    """Construct the chat messages with retrieved context injected (traced)."""
    context = "\n\n".join(
        f"[{doc['id']}] {doc['text']}" for doc in retrieved_docs
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT.format(context=context)},
        {"role": "user", "content": question},
    ]

### 4d — Traced RAG query

The top-level function uses `span_type=SpanType.CHAIN` to group the entire pipeline into one trace. Every sub-function creates a child span automatically.

In [ ]:
@mlflow.trace(span_type=SpanType.CHAIN)
def rag_query_traced(question: str, index: list[dict], top_k: int = 3) -> str:
    """End-to-end RAG with full MLflow tracing."""
    # Retrieve — creates a RETRIEVER span
    retrieved_docs = semantic_search_traced(question, index, top_k=top_k)

    # Augment — creates a CHAIN span
    messages = build_prompt_traced(question, retrieved_docs)

    # Generate — creates an LLM span via start_span
    with mlflow.start_span(name="llm_generate", span_type=SpanType.LLM) as span:
        span.set_inputs({"messages": messages, "model": MODEL})
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.2,
            max_completion_tokens=400,
        )
        answer = response.choices[0].message.content
        span.set_outputs({"answer": answer})
        span.set_attributes({
            "token_count.input": response.usage.prompt_tokens,
            "token_count.output": response.usage.completion_tokens,
        })

    return answer

---
## 5 — Run the traced pipeline

Same questions as before — but now every step is visible in the MLflow UI. Open the **Traces** tab to see the full span tree.

In [ ]:
test_questions = [
    "How do I trace my LLM calls in MLflow?",
    "What is the AI Gateway and what does it do?",
    "How do I evaluate my agent's responses?",
    "What is judge alignment and how does it work?",
    "How do I version my prompts in MLflow?",
]

for q in test_questions:
    print(f"Q: {q}")
    answer = rag_query_traced(q, doc_index)
    print(f"A: {answer[:200]}...\n")

---
## 6 — What tracing reveals

Open the MLflow UI (`http://127.0.0.1:5000`) and click on any trace. You should see:

```
rag_query_traced (CHAIN)
  |-- semantic_search_traced (RETRIEVER)
  |     |-- get_embeddings_traced (EMBEDDING)
  |-- build_prompt_traced (CHAIN)
  |-- llm_generate (LLM)
```

For each span you can inspect:
- **Inputs/Outputs** — what went in and what came out
- **Latency** — how long each step took
- **Token counts** — on the LLM span
- **Retrieved docs** — on the RETRIEVER span, so you can check relevance

This is the foundation for debugging and evaluation. In the next notebook, we'll use these traces to run scorers like `RetrievalGroundedness` that check whether the answer is actually supported by the retrieved context.